# Disco — Quickstart

This notebook walks through a complete Disco run: uploading a dataset, waiting for the analysis to finish, and working with the results.

**What Disco does:** Finds novel, statistically validated patterns in tabular data — feature interactions, subgroup effects, and conditional relationships you would not think to look for. Each pattern is validated on hold-out data with FDR-corrected p-values and checked against academic literature for novelty.

Get an API key at [disco.leap-labs.com/developers](https://disco.leap-labs.com/developers) or sign up below.

In [6]:
# Install the SDK (run once)
%pip install "discovery-engine-api[jupyter]" pandas --quiet

Note: you may need to restart the kernel to use updated packages.


## Load Your Data

Disco works with any tabular dataset. You need:
- A **target column** — the outcome you want to understand (e.g. treatment response, churn, yield)
- **Feature columns** — everything that might be related to it

Below we create a synthetic clinical dataset for demonstration purposes. Replace this with your own dataset!

In [7]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)
n = 500

age = rng.integers(25, 75, n)
bmi = rng.normal(27, 5, n).clip(16, 45)
blood_pressure = rng.normal(120, 18, n).clip(80, 180)
cholesterol = rng.normal(200, 40, n).clip(120, 320)
smoking = rng.choice(["never", "former", "current"], n, p=[0.5, 0.3, 0.2])
exercise_days_per_week = rng.integers(0, 8, n)
diet_score = rng.integers(1, 11, n)  # 1–10

# Simulate a target with real structure baked in:
# Older patients with high BP and low exercise respond poorly
treatment_response = (
    50
    - 0.3 * np.maximum(age - 55, 0)
    - 0.2 * np.maximum(blood_pressure - 130, 0)
    + 2 * exercise_days_per_week
    + 1.5 * diet_score
    + rng.normal(0, 8, n)
).clip(0, 100)

df = pd.DataFrame(
    {
        "patient_id": [f"P{i:04d}" for i in range(n)],
        "age": age,
        "bmi": bmi.round(1),
        "blood_pressure": blood_pressure.round(0).astype(int),
        "cholesterol": cholesterol.round(0).astype(int),
        "smoking": smoking,
        "exercise_days_per_week": exercise_days_per_week,
        "diet_score": diet_score,
        "treatment_response": treatment_response.round(1),
    }
)

print(df.shape)
df.head()

(500, 9)


,patient_id,age,bmi,blood_pressure,cholesterol,smoking,exercise_days_per_week,diet_score,treatment_response
0,P0000,29,26.5,141,160,never,2,1,39.3
1,P0001,63,18.2,102,267,current,1,10,66.0
2,P0002,57,19.7,87,179,never,5,10,69.7
3,P0003,46,37.6,122,242,former,0,6,56.0
4,P0004,46,20.6,137,203,current,3,2,50.4


## Run Disco

Pass the DataFrame directly. `engine.run()` submits the job and waits for completion (runs take 3–15 minutes).

Setting `visibility="public"` makes the run free — results are published to the public gallery. Use `visibility="private"` to keep results private (costs credits).

Provide `column_descriptions` for any non-obvious column names — it significantly improves pattern explanations.

## Get an API Key

Get a free API key at [disco.leap-labs.com/developers](https://disco.leap-labs.com/developers) and paste it into the cell below.

In [ ]:
from discovery import Engine

api_key = "YOUR API KEY GOES HERE"  # Replace with your key

engine = Engine(api_key=api_key)

result = engine.run(
    file=df,
    target_column="treatment_response",
    title="Clinical trial response dataset",
    description="Synthetic dataset modelling treatment response in a clinical cohort.",
    column_descriptions={
        "bmi": "Body mass index",
        "blood_pressure": "Systolic blood pressure (mmHg)",
        "cholesterol": "Total cholesterol (mg/dL)",
        "exercise_days_per_week": "Number of days per week with ≥30 min exercise",
        "diet_score": "Self-reported diet quality score (1=poor, 10=excellent)",
        "treatment_response": "Treatment response score (0–100, higher is better)",
    },
    excluded_columns=["patient_id"],  # Exclude identifiers
    visibility="public",
    wait=True,
)

## Explore the Results

Results contain **patterns** — each is a combination of conditions (feature ranges/values) that together define a subgroup with a meaningfully different outcome. Not single correlations.

In [9]:
# High-level summary
if result.summary:
    print(result.summary.overview)
    print()
    for insight in result.summary.key_insights:
        print(f"  • {insight}")

Discovery Engine found that lifestyle factors show the strongest associations with treatment response, with exercise frequency and diet quality being the most predictive features. Pattern 1, showing the largest effect size, reveals that patients with cholesterol levels in the normal range (120-284 mg/dL) are associated with a 0.22-point decrease in treatment response compared to baseline. All three patterns confirm established relationships in healthcare literature, with exercise frequency (0-4 days/week) associated with reduced treatment response by 3.66 points and higher diet scores (3-10) linked to improved response by 1.50 points.

  • Lifestyle factors dominate treatment response prediction, with exercise frequency showing the strongest association (correlation: 0.481) and accounting for 42.48% of feature importance [Pattern 2]
  • Diet quality emerges as the second most predictive factor, with scores between 3-10 associated with 1.50-point improvements in treatment response and 3

In [10]:
# Novel patterns — findings not in the existing literature
novel = [p for p in result.patterns if p.novelty_type == "novel" and p.p_value < 0.05]

print(f"{len(novel)} novel patterns (p < 0.05)\n")

for p in novel[:5]:
    print(f"Pattern: {p.description}")
    print(
        f"  p={p.p_value:.4f} | effect={p.abs_target_change:+.2f} | support={p.support_percentage:.1f}%"
    )
    print(f"  Why novel: {p.novelty_explanation}")
    if p.citations:
        print(f"  Checked against {len(p.citations)} paper(s)")
    print()

0 novel patterns (p < 0.05)



In [11]:
# Inspect the conditions of the top pattern
if novel:
    top = novel[0]
    print(f"Top pattern: {top.description}\n")
    for cond in top.conditions:
        if cond["type"] == "continuous":
            print(f"  {cond['feature']}: {cond['min_value']} — {cond['max_value']}")
        elif cond["type"] == "categorical":
            print(f"  {cond['feature']}: {cond['values']}")

In [12]:
# Feature importance scores are signed: positive = increases prediction, negative = decreases it
if result.feature_importance:
    scores = sorted(
        result.feature_importance.scores,
        key=lambda s: abs(s.score),
        reverse=True,
    )
    print("Feature importance (signed):")
    for s in scores:
        bar = "█" * int(abs(s.score) * 20)
        direction = "+" if s.score >= 0 else "-"
        print(f"  {s.feature:<30} {direction}{bar}")

Feature importance (signed):
  exercise_days_per_week         +████████
  diet_score                     +██████
  age                            +█
  blood_pressure                 +█
  cholesterol                    +
  bmi                            +
  smoking                        +


In [13]:
# Interactive report — share this link to explore results visually
print(f"Interactive report: {result.report_url}")

Interactive report: https://disco.leap-labs.com/reports/019d0c6e-dbb2-77c7-9f98-515a1fe2a8bb


## Next Steps

- **Deeper analysis:** Upgrade to see more patterns. 
- **Full SDK reference:** https://github.com/leap-laboratories/discovery-engine/blob/main/docs/python-sdk.md